In [1]:
pip install torch torchvision scikit-learn matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

Pre Processing

In [3]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
])

In [4]:
DATASET_DIR = "CAT_SKIN_SPLIT"

train_dataset = ImageFolder(f"{DATASET_DIR}/train", transform=transform)
val_dataset   = ImageFolder(f"{DATASET_DIR}/val",   transform=transform)
test_dataset  = ImageFolder(f"{DATASET_DIR}/test",  transform=transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=16)
test_loader  = DataLoader(test_dataset,  batch_size=16)


CNN

In [5]:
model = models.resnet50(pretrained=True)

c:\Users\chanu\OneDrive - Informatics Institute of Technology\Desktop\IRP\Cat Skin DIsease Classification\.venv\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\chanu\OneDrive - Informatics Institute of Technology\Desktop\IRP\Cat Skin DIsease Classification\.venv\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [6]:
# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Unfreeze the last ResNet block
for param in model.layer4.parameters():
    param.requires_grad = True

In [7]:
num_classes = len(train_dataset.classes)

model.fc = nn.Linear(model.fc.in_features, num_classes)

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.01)


In [ ]:
import numpy as np
best_val_loss    = np.inf
patience         = 5
patience_counter = 0
best_model_state = None

num_epochs = 30

for epoch in range(num_epochs):

    model.train()
    running_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}")

    # val loss check inside the training loop
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_loss += criterion(outputs, labels).item()
    val_loss /= len(val_loader)

    print(f"         Val Loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_counter = 0
        best_model_state = model.state_dict().copy()
        print(f"         ✓ Saved best model")
    else:
        patience_counter += 1
        print(f"         ✗ No improvement ({patience_counter}/{patience})")
        if patience_counter >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

    
# Restore best weights before evaluation
model.load_state_dict(best_model_state)

Epoch 1, Loss: 2.1257
         Val Loss: 1.1909
         ✓ Saved best model
Epoch 2, Loss: 0.8686
         Val Loss: 0.4385
         ✓ Saved best model
Epoch 3, Loss: 0.9602
         Val Loss: 0.6634
         ✗ No improvement (1/5)
Epoch 4, Loss: 0.7822
         Val Loss: 0.5441
         ✗ No improvement (2/5)
Epoch 5, Loss: 0.9185
         Val Loss: 1.2518
         ✗ No improvement (3/5)
Epoch 6, Loss: 0.6238
         Val Loss: 0.4138
         ✓ Saved best model
Epoch 7, Loss: 0.5838
         Val Loss: 0.5394
         ✗ No improvement (1/5)
Epoch 8, Loss: 0.4413
         Val Loss: 0.4658
         ✗ No improvement (2/5)
Epoch 9, Loss: 0.4634
         Val Loss: 0.7679
         ✗ No improvement (3/5)
Epoch 10, Loss: 0.5052
         Val Loss: 0.6056
         ✗ No improvement (4/5)
Epoch 11, Loss: 0.6720
         Val Loss: 0.5573
         ✗ No improvement (5/5)

Early stopping at epoch 11


<All keys matched successfully>

In [10]:
model.eval()
correct = 0
total = 0

with torch.no_grad():

    for images, labels in val_loader:

        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Validation Accuracy:", correct / total)

Validation Accuracy: 0.8493150684931506


In [11]:
model.eval()
correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Test Accuracy:", correct / total)

Test Accuracy: 0.777027027027027
